# Model Implementation

This notebook reproduces the baseline architectures described in the paper.

We implement six models:

1. FFT-MLP (All Sensors)
2. FFT-MLP (IMU + Thermopile)
3. CNN-BiLSTM (TOF only)
4. Late Fusion (Model 2 + Model 3)
5. Intermediate Fusion
6. FFT + Random Forest

All models are trained using sequence-level splits generated in the preprocessing notebook.

## 1. Environment setup 

We import all necessary libraries and set random seeds for reproducibility.

In [2]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


## Imports and loaded files 

Load the preprocessed datasets and feature metadata.


In [3]:
DATA_PROCESSED = Path("../data/processed")

with open(DATA_PROCESSED / "train_clean.pkl", "rb") as f:
    train_df = pickle.load(f)

with open(DATA_PROCESSED / "test_clean.pkl", "rb") as f:
    test_df = pickle.load(f)

with open(DATA_PROCESSED / "feature_map.pkl", "rb") as f:
    fmap = pickle.load(f)

IMU_COLS = fmap["imu_cols"]
THM_COLS = fmap["thm_cols"]
TOF_COLS = fmap["tof_cols"]
ALL_SENSORS = fmap["feature_cols"]   # 332
LABEL_COL = fmap["label_col"]        # "gesture"

print("Train sequences:", train_df["sequence_id"].nunique(), "rows:", train_df.shape)
print("Test sequences :", test_df["sequence_id"].nunique(),  "rows:", test_df.shape)

# Safety: force numeric (prevents NaNs during FFT conversion)
train_df[ALL_SENSORS] = train_df[ALL_SENSORS].apply(pd.to_numeric, errors="coerce").fillna(-1.0)
test_df[ALL_SENSORS]  = test_df[ALL_SENSORS].apply(pd.to_numeric, errors="coerce").fillna(-1.0)

print("NaNs train sensors:", train_df[ALL_SENSORS].isna().sum().sum())
print("NaNs test sensors :", test_df[ALL_SENSORS].isna().sum().sum())


Train sequences: 6515 rows: (460717, 341)
Test sequences : 1629 rows: (113993, 341)
NaNs train sensors: 0
NaNs test sensors : 0


## 4.2 Label Encoding

Gesture labels are encoded into numerical format
using scikit-learn's LabelEncoder.

The number of unique gesture classes is determined here.

In [4]:
le = LabelEncoder()
le.fit(train_df[LABEL_COL].astype(str).values)
num_classes = len(le.classes_)
print("Num gesture classes:", num_classes)  # should be 18


Num gesture classes: 18


## Sequence Grouping and FFT Preparation

Here we:

- Group data by `sequence_id`
- Keep temporal order using `sequence_counter`
- Pad or truncate sequences to a fixed length
- Apply FFT to each sequence
- Flatten the magnitude spectrum for model input

The maximum sequence length is chosen from the training distribution
(90th percentile) to keep inputs consistent.


In [5]:
def make_groups(df: pd.DataFrame):
    return dict(tuple(df.groupby("sequence_id")))

train_groups = make_groups(train_df)
test_groups  = make_groups(test_df)

train_ids = list(train_groups.keys())
test_ids  = list(test_groups.keys())

def pad_or_truncate(ts: np.ndarray, max_len: int) -> np.ndarray:
    T, F = ts.shape
    if T >= max_len:
        return ts[:max_len]
    out = np.zeros((max_len, F), dtype=ts.dtype)
    out[:T] = ts
    return out

def seq_to_fft_vector(seq_df: pd.DataFrame, feature_cols: list, max_len: int) -> np.ndarray:
    # ensure temporal order
    if "sequence_counter" in seq_df.columns:
        seq_df = seq_df.sort_values("sequence_counter")

    ts = seq_df[feature_cols].to_numpy(dtype=np.float32)
    ts = np.nan_to_num(ts, nan=-1.0, posinf=0.0, neginf=0.0)
    ts = pad_or_truncate(ts, max_len)

    fft_vals = np.fft.fft(ts, axis=0)
    fft_mag = np.abs(fft_vals).astype(np.float32)
    return fft_mag.flatten()

# Choose a fixed length from training distribution
train_seq_lens = train_df.groupby("sequence_id").size().values
MAX_LEN = int(np.percentile(train_seq_lens, 90))
MAX_LEN = max(32, min(MAX_LEN, 256))
print("MAX_LEN:", MAX_LEN)


MAX_LEN: 102


## FFT Dataset and DataLoader

We define a custom PyTorch Dataset for the FFT-MLP models.

Each sample:
- Converts one sequence into an FFT feature vector
- Encodes the gesture label
- Returns tensors for training

We use a batch size of 256 as described in the paper.


In [6]:
class FFTMLPDataset(Dataset):
    def __init__(self, grouped, seq_ids, feature_cols, label_col, label_encoder, max_len):
        self.grouped = grouped
        self.seq_ids = seq_ids
        self.feature_cols = feature_cols
        self.label_col = label_col
        self.le = label_encoder
        self.max_len = max_len

    def __len__(self):
        return len(self.seq_ids)

    def __getitem__(self, idx):
        sid = self.seq_ids[idx]
        s = self.grouped[sid]
        x = seq_to_fft_vector(s, self.feature_cols, self.max_len)
        y = self.le.transform([str(s[self.label_col].iloc[0])])[0]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

BATCH_SIZE = 256


## Normalization Utilities

For FFT-based models, we compute mean and standard deviation
from the training data and apply normalization during training
and evaluation.

This helps stabilize training and prevents numerical issues.


In [7]:
@torch.no_grad()
def compute_norm_stats(loader, max_batches=30, eps=1e-6):
    xs = []
    for i, (x, _) in enumerate(loader):
        xs.append(x.float())
        if i + 1 >= max_batches:
            break
    X = torch.cat(xs, dim=0)
    X = torch.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    mu = X.mean(dim=0)
    sigma = X.std(dim=0)
    sigma = torch.clamp(sigma, min=eps)
    return mu, sigma

@torch.no_grad()
def get_logits_and_labels(model, loader, mu=None, sigma=None):
    model.eval()
    all_logits, all_y = [], []
    for x, y in loader:
        x = x.to(device).float()
        if mu is not None and sigma is not None:
            x = (x - mu) / sigma
            x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        logits = model(x)
        all_logits.append(logits.cpu())
        all_y.append(y.cpu())
    return torch.cat(all_logits), torch.cat(all_y)


## Model 1: FFT-MLP (All Sensors)
FFT magnitude features from all 332 sensors, followed by an MLP (128→64→64). 


In [8]:
m1_features = ALL_SENSORS
m1_input_dim = MAX_LEN * len(m1_features)

train_loader_m1 = DataLoader(
    FFTMLPDataset(train_groups, train_ids, m1_features, LABEL_COL, le, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
test_loader_m1 = DataLoader(
    FFTMLPDataset(test_groups, test_ids, m1_features, LABEL_COL, le, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print("M1 input dim:", m1_input_dim)


M1 input dim: 33864


In [9]:
class FFTMLP(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)

def train_mlp(model, train_loader, test_loader, epochs=20, lr=0.001):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    mu, sigma = compute_norm_stats(train_loader, max_batches=30)
    mu, sigma = mu.to(device), sigma.to(device)

    def train_one_epoch():
        model.train()
        total_loss, total, correct = 0.0, 0, 0
        for x, y in train_loader:
            x = x.to(device).float()
            y = y.to(device)

            x = (x - mu) / sigma
            x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

            opt.zero_grad()
            logits = model(x)
            loss = crit(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item() * x.size(0)
            total += x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
        return total_loss / total, correct / total

    @torch.no_grad()
    def evaluate():
        model.eval()
        total, correct = 0, 0
        y_true, y_pred = [], []
        for x, y in test_loader:
            x = x.to(device).float()
            y = y.to(device)
            x = (x - mu) / sigma
            x = torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

            logits = model(x)
            pred = logits.argmax(1)

            total += x.size(0)
            correct += (pred == y).sum().item()
            y_true.append(y.cpu().numpy())
            y_pred.append(pred.cpu().numpy())

        y_true = np.concatenate(y_true)
        y_pred = np.concatenate(y_pred)
        acc = correct / total
        macro = f1_score(y_true, y_pred, average="macro")
        return acc, macro

    for ep in range(1, epochs + 1):
        loss, tr_acc = train_one_epoch()
        te_acc, te_macro = evaluate()
        print(f"Epoch {ep:02d} | loss={loss:.4f} | train_acc={tr_acc:.4f} | test_acc={te_acc:.4f} | macroF1={te_macro:.4f}")

    return model, mu, sigma

model1 = FFTMLP(m1_input_dim, num_classes)
model1, mu1, sig1 = train_mlp(model1, train_loader_m1, test_loader_m1, epochs=20, lr=0.001)


Epoch 01 | loss=2.3379 | train_acc=0.2625 | test_acc=0.3407 | macroF1=0.2963
Epoch 02 | loss=1.5254 | train_acc=0.5010 | test_acc=0.4168 | macroF1=0.4020
Epoch 03 | loss=0.9974 | train_acc=0.6856 | test_acc=0.4279 | macroF1=0.4296
Epoch 04 | loss=0.6024 | train_acc=0.8281 | test_acc=0.4309 | macroF1=0.4370
Epoch 05 | loss=0.3655 | train_acc=0.9013 | test_acc=0.4359 | macroF1=0.4382
Epoch 06 | loss=0.2364 | train_acc=0.9366 | test_acc=0.4346 | macroF1=0.4303
Epoch 07 | loss=0.1813 | train_acc=0.9536 | test_acc=0.4279 | macroF1=0.4417
Epoch 08 | loss=0.1676 | train_acc=0.9566 | test_acc=0.4518 | macroF1=0.4499
Epoch 09 | loss=0.1271 | train_acc=0.9642 | test_acc=0.4432 | macroF1=0.4321
Epoch 10 | loss=0.1099 | train_acc=0.9699 | test_acc=0.4457 | macroF1=0.4486
Epoch 11 | loss=0.1165 | train_acc=0.9667 | test_acc=0.4690 | macroF1=0.4736
Epoch 12 | loss=0.1050 | train_acc=0.9716 | test_acc=0.4420 | macroF1=0.4491
Epoch 13 | loss=0.1029 | train_acc=0.9713 | test_acc=0.4487 | macroF1=0.4338

In [10]:
test_logits_m1, test_y = get_logits_and_labels(model1, test_loader_m1, mu1, sig1)
test_pred_m1 = test_logits_m1.argmax(dim=1)

torch.save(
    {"logits": test_logits_m1, "y_true": test_y, "y_pred": test_pred_m1, "classes": le.classes_},
    DATA_PROCESSED / "model1_fft_mlp_all_outputs.pt"
)
print("Saved model1_fft_mlp_all_outputs.pt")


Saved model1_fft_mlp_all_outputs.pt


### Model 1 Summary

The FFT-MLP model trained on all sensors serves as the baseline
deep learning architecture for multi-sensor frequency-domain classification.


## Model 2: FFT-MLP (IMU + THM Only)
Same pipeline as Model 1, but using only IMU+THM (12 features).


In [11]:
m2_features = IMU_COLS + THM_COLS
m2_input_dim = MAX_LEN * len(m2_features)

train_loader_m2 = DataLoader(
    FFTMLPDataset(train_groups, train_ids, m2_features, LABEL_COL, le, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
test_loader_m2 = DataLoader(
    FFTMLPDataset(test_groups, test_ids, m2_features, LABEL_COL, le, MAX_LEN),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

model2 = FFTMLP(m2_input_dim, num_classes)
model2, mu2, sig2 = train_mlp(model2, train_loader_m2, test_loader_m2, epochs=20, lr=0.001)

test_logits_m2, test_y2 = get_logits_and_labels(model2, test_loader_m2, mu2, sig2)
test_pred_m2 = test_logits_m2.argmax(dim=1)

torch.save(
    {"logits": test_logits_m2, "y_true": test_y2, "y_pred": test_pred_m2, "classes": le.classes_},
    DATA_PROCESSED / "model2_fft_mlp_imu_thm_outputs.pt"
)
print("Saved model2_fft_mlp_imu_thm_outputs.pt")


Epoch 01 | loss=2.6466 | train_acc=0.1418 | test_acc=0.2394 | macroF1=0.1691
Epoch 02 | loss=2.1806 | train_acc=0.2783 | test_acc=0.2861 | macroF1=0.2385
Epoch 03 | loss=1.8995 | train_acc=0.3521 | test_acc=0.3505 | macroF1=0.3128
Epoch 04 | loss=1.7101 | train_acc=0.4072 | test_acc=0.3689 | macroF1=0.3593
Epoch 05 | loss=1.5884 | train_acc=0.4482 | test_acc=0.3800 | macroF1=0.3879
Epoch 06 | loss=1.4818 | train_acc=0.4804 | test_acc=0.3874 | macroF1=0.4100
Epoch 07 | loss=1.3803 | train_acc=0.5245 | test_acc=0.3953 | macroF1=0.4096
Epoch 08 | loss=1.2958 | train_acc=0.5529 | test_acc=0.3837 | macroF1=0.4067
Epoch 09 | loss=1.2271 | train_acc=0.5741 | test_acc=0.3769 | macroF1=0.4115
Epoch 10 | loss=1.1518 | train_acc=0.6026 | test_acc=0.3769 | macroF1=0.4086
Epoch 11 | loss=1.0799 | train_acc=0.6312 | test_acc=0.3880 | macroF1=0.4104
Epoch 12 | loss=1.0210 | train_acc=0.6574 | test_acc=0.3806 | macroF1=0.4137
Epoch 13 | loss=0.9569 | train_acc=0.6787 | test_acc=0.3726 | macroF1=0.3927

### Model 2 Summary

This model evaluates the performance contribution
of non-TOF sensor modalities.

## Model 3: CNN-BiLSTM (TOF Only)
We reshape TOF features per timestep to an 8×8 "image" per sensor (5 TOF sensors total),
then use a CNN to extract features per timestep and a BiLSTM to model temporal dynamics.


In [12]:
def tof_row_to_5x8x8(row_tof: np.ndarray) -> np.ndarray:
    # row_tof shape: [320] = 5 sensors × 64 pixels
    # output: [5, 8, 8]
    return row_tof.reshape(5, 8, 8)

class TOFSequenceDataset(Dataset):
    def __init__(self, grouped, seq_ids, tof_cols, label_col, label_encoder, max_len):
        self.grouped = grouped
        self.seq_ids = seq_ids
        self.tof_cols = tof_cols
        self.label_col = label_col
        self.le = label_encoder
        self.max_len = max_len

    def __len__(self):
        return len(self.seq_ids)

    def __getitem__(self, idx):
        sid = self.seq_ids[idx]
        s = self.grouped[sid]
        if "sequence_counter" in s.columns:
            s = s.sort_values("sequence_counter")

        ts = s[self.tof_cols].to_numpy(dtype=np.float32)  # [T, 320]
        ts = np.nan_to_num(ts, nan=-1.0, posinf=0.0, neginf=0.0)

        # pad/truncate time
        ts = pad_or_truncate(ts, self.max_len)            # [L, 320]

        # reshape each timestep to [5,8,8]
        imgs = ts.reshape(self.max_len, 5, 8, 8)          # [L, 5, 8, 8]

        y = self.le.transform([str(s[self.label_col].iloc[0])])[0]
        return torch.from_numpy(imgs), torch.tensor(y, dtype=torch.long)

train_loader_m3 = DataLoader(
    TOFSequenceDataset(train_groups, train_ids, TOF_COLS, LABEL_COL, le, MAX_LEN),
    batch_size=64, shuffle=True, num_workers=0
)
test_loader_m3 = DataLoader(
    TOFSequenceDataset(test_groups, test_ids, TOF_COLS, LABEL_COL, le, MAX_LEN),
    batch_size=64, shuffle=False, num_workers=0
)

print("M3 batches:", len(train_loader_m3), len(test_loader_m3))


M3 batches: 102 26


In [13]:
class CNNBiLSTM(nn.Module):
    def __init__(self, num_classes: int, cnn_out=64, lstm_hidden=64):
        super().__init__()
        # small CNN over 5-channel 8x8
        self.cnn = nn.Sequential(
            nn.Conv2d(5, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 4x4
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),          # 2x2
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, cnn_out),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(
            input_size=cnn_out,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=True
        )
        self.head = nn.Linear(2 * lstm_hidden, num_classes)

    def forward(self, x):
        # x: [B, L, 5, 8, 8]
        B, L, C, H, W = x.shape
        x = x.view(B * L, C, H, W)
        feat = self.cnn(x)                 # [B*L, cnn_out]
        feat = feat.view(B, L, -1)         # [B, L, cnn_out]
        out, _ = self.lstm(feat)           # [B, L, 2*h]
        last = out[:, -1, :]               # last timestep
        return self.head(last)             # [B, num_classes]

def train_cnn_bilstm(model, train_loader, test_loader, epochs=20, lr=0.001):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    def train_one_epoch():
        model.train()
        total_loss, total, correct = 0.0, 0, 0
        for x, y in train_loader:
            x = x.to(device).float()
            y = y.to(device)

            opt.zero_grad()
            logits = model(x)
            loss = crit(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item() * x.size(0)
            total += x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
        return total_loss / total, correct / total

    @torch.no_grad()
    def evaluate():
        model.eval()
        total, correct = 0, 0
        y_true, y_pred = [], []
        all_logits = []
        for x, y in test_loader:
            x = x.to(device).float()
            y = y.to(device)
            logits = model(x)
            pred = logits.argmax(1)

            total += x.size(0)
            correct += (pred == y).sum().item()
            y_true.append(y.cpu().numpy())
            y_pred.append(pred.cpu().numpy())
            all_logits.append(logits.cpu())

        y_true = np.concatenate(y_true)
        y_pred = np.concatenate(y_pred)
        acc = correct / total
        macro = f1_score(y_true, y_pred, average="macro")
        return acc, macro, torch.cat(all_logits, dim=0)

    for ep in range(1, epochs + 1):
        loss, tr_acc = train_one_epoch()
        te_acc, te_macro, _ = evaluate()
        print(f"Epoch {ep:02d} | loss={loss:.4f} | train_acc={tr_acc:.4f} | test_acc={te_acc:.4f} | macroF1={te_macro:.4f}")

    # final logits on test
    _, _, test_logits = evaluate()
    return model, test_logits

model3 = CNNBiLSTM(num_classes=num_classes)
model3, test_logits_m3 = train_cnn_bilstm(model3, train_loader_m3, test_loader_m3, epochs=20, lr=0.001)

test_pred_m3 = test_logits_m3.argmax(dim=1)
# Note: test_loader_m3 yields y too, but easiest is to reload from model2's saved y_true order ONLY if order matches.
# We'll save y_true from a pass:
@torch.no_grad()
def collect_y_true(loader):
    ys=[]
    for _, y in loader:
        ys.append(y.cpu())
    return torch.cat(ys, dim=0)

test_y3 = collect_y_true(test_loader_m3)

torch.save(
    {"logits": test_logits_m3, "y_true": test_y3, "y_pred": test_pred_m3, "classes": le.classes_},
    DATA_PROCESSED / "model3_cnn_bilstm_tof_outputs.pt"
)
print("Saved model3_cnn_bilstm_tof_outputs.pt")


Epoch 01 | loss=2.7675 | train_acc=0.0878 | test_acc=0.0994 | macroF1=0.0353
Epoch 02 | loss=2.6100 | train_acc=0.1325 | test_acc=0.1657 | macroF1=0.0739
Epoch 03 | loss=2.4726 | train_acc=0.1544 | test_acc=0.1780 | macroF1=0.0922
Epoch 04 | loss=2.3778 | train_acc=0.1771 | test_acc=0.1793 | macroF1=0.0830
Epoch 05 | loss=2.3258 | train_acc=0.1808 | test_acc=0.1762 | macroF1=0.0736
Epoch 06 | loss=2.2804 | train_acc=0.1891 | test_acc=0.2026 | macroF1=0.1197
Epoch 07 | loss=2.2642 | train_acc=0.1946 | test_acc=0.1983 | macroF1=0.1143
Epoch 08 | loss=2.2333 | train_acc=0.2075 | test_acc=0.2118 | macroF1=0.1349
Epoch 09 | loss=2.2135 | train_acc=0.2216 | test_acc=0.2038 | macroF1=0.1200
Epoch 10 | loss=2.1979 | train_acc=0.2244 | test_acc=0.2566 | macroF1=0.1808
Epoch 11 | loss=2.1414 | train_acc=0.2665 | test_acc=0.2498 | macroF1=0.1584
Epoch 12 | loss=2.0938 | train_acc=0.2698 | test_acc=0.2413 | macroF1=0.1583
Epoch 13 | loss=2.0632 | train_acc=0.2935 | test_acc=0.2787 | macroF1=0.1945

### Model 3 Summary

The CNN-BiLSTM model evaluates the effectiveness
of spatial-only sensing for gesture recognition.


## Model 4: Late Fusion Ensemble
Combine logits from Model 2 (FFT-MLP IMU+THM) and Model 3 (CNN-BiLSTM TOF):
`logits = 0.3 * logits_mlp + 0.7 * logits_cnn`


In [14]:
m2 = torch.load(DATA_PROCESSED / "model2_fft_mlp_imu_thm_outputs.pt", weights_only=False)
m3 = torch.load(DATA_PROCESSED / "model3_cnn_bilstm_tof_outputs.pt", weights_only=False)

logits2 = m2["logits"]
logits3 = m3["logits"]

# Ensure aligned ordering (same test_ids order). If your loaders differ in order, we'll fix by storing sequence_ids in each output.
assert logits2.shape == logits3.shape, "Logit shapes differ — ordering/alignment needs fix."

late_logits = 0.3 * logits2 + 0.7 * logits3
late_pred = late_logits.argmax(dim=1)

y_true = m2["y_true"]  # assume same
late_macro18 = f1_score(y_true.numpy(), late_pred.numpy(), average="macro")
print("Late fusion macro-F1 (18):", late_macro18)

late_acc = (late_pred == y_true).float().mean().item()
print("Late fusion accuracy:", late_acc)

torch.save(
    {"logits": late_logits, "y_true": y_true, "y_pred": late_pred, "classes": m2["classes"]},
    DATA_PROCESSED / "model4_late_fusion_outputs.pt"
)
print("Saved model4_late_fusion_outputs.pt")


Late fusion macro-F1 (18): 0.4849561710605415
Late fusion accuracy: 0.46899938583374023
Saved model4_late_fusion_outputs.pt


## Model 5: Intermediate Fusion (IMU+THM FFT encoder + TOF CNN-BiLSTM encoder)

We learn two feature embeddings:
- IMU+THM branch: FFT → MLP encoder → 64-dim embedding
- TOF branch: CNN → BiLSTM → 128-dim embedding

We concatenate embeddings (64 + 128) and train a classifier head for 18 gesture classes.


In [15]:
## Feature sets + MAX_LEN 

M5_IMU_THM = IMU_COLS + THM_COLS     # 12
M5_TOF = TOF_COLS                   #320
MAX_LEN_M5 = MAX_LEN                
print("MAX_LEN_M5:", MAX_LEN_M5)



MAX_LEN_M5: 102


## Paired dataset (same sequence_id yields both inputs)
This dataset returns:
x_fft for IMU+THM
x_tof for TOF images
label y

In [16]:
class IntermediateFusionDataset(Dataset):
    def __init__(self, grouped, seq_ids, imu_thm_cols, tof_cols, label_col, label_encoder, max_len):
        self.grouped = grouped
        self.seq_ids = seq_ids
        self.imu_thm_cols = imu_thm_cols
        self.tof_cols = tof_cols
        self.label_col = label_col
        self.le = label_encoder
        self.max_len = max_len

    def __len__(self):
        return len(self.seq_ids)

    def __getitem__(self, idx):
        sid = self.seq_ids[idx]
        s = self.grouped[sid]

        # enforce time order
        if "sequence_counter" in s.columns:
            s = s.sort_values("sequence_counter")

        # ---- IMU+THM FFT vector ----
        x_fft = seq_to_fft_vector(s, self.imu_thm_cols, self.max_len)   # [L*12]

        # ---- TOF sequence images ----
        ts_tof = s[self.tof_cols].to_numpy(dtype=np.float32)            # [T, 320]
        ts_tof = np.nan_to_num(ts_tof, nan=-1.0, posinf=0.0, neginf=0.0)
        ts_tof = pad_or_truncate(ts_tof, self.max_len)                  # [L, 320]
        x_tof = ts_tof.reshape(self.max_len, 5, 8, 8)                   # [L, 5, 8, 8]

        # label
        y = self.le.transform([str(s[self.label_col].iloc[0])])[0]

        return (
            torch.from_numpy(x_fft),
            torch.from_numpy(x_tof),
            torch.tensor(y, dtype=torch.long),
        )

# loaders
BATCH_M5 = 64  # CNN+LSTM on CPU -> keep moderate

train_loader_m5 = DataLoader(
    IntermediateFusionDataset(train_groups, train_ids, M5_IMU_THM, M5_TOF, LABEL_COL, le, MAX_LEN_M5),
    batch_size=BATCH_M5, shuffle=True, num_workers=0
)

test_loader_m5 = DataLoader(
    IntermediateFusionDataset(test_groups, test_ids, M5_IMU_THM, M5_TOF, LABEL_COL, le, MAX_LEN_M5),
    batch_size=BATCH_M5, shuffle=False, num_workers=0
)

m5_input_dim_fft = MAX_LEN_M5 * len(M5_IMU_THM)
print("M5 FFT input dim:", m5_input_dim_fft)


M5 FFT input dim: 1224


In [17]:
class IMUTHMEncoder(nn.Module):
    # FFT vector -> 64-dim embedding
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)  # [B, 64]

class TOFEncoder(nn.Module):
    # TOF images -> BiLSTM -> 128-dim embedding
    def __init__(self, cnn_out=64, lstm_hidden=64):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(5, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),              # 4x4
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),              # 2x2
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, cnn_out),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(
            input_size=cnn_out,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        # x: [B, L, 5, 8, 8]
        B, L, C, H, W = x.shape
        x = x.view(B * L, C, H, W)
        feat = self.cnn(x)            # [B*L, cnn_out]
        feat = feat.view(B, L, -1)    # [B, L, cnn_out]
        out, _ = self.lstm(feat)      # [B, L, 2*h]
        return out[:, -1, :]          # [B, 2*h] = [B, 128]

class IntermediateFusionModel(nn.Module):
    def __init__(self, fft_input_dim: int, num_classes: int):
        super().__init__()
        self.imu_enc = IMUTHMEncoder(fft_input_dim)    # 64
        self.tof_enc = TOFEncoder()                    # 128
        self.head = nn.Sequential(
            nn.Linear(64 + 128, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x_fft, x_tof):
        e1 = self.imu_enc(x_fft)
        e2 = self.tof_enc(x_tof)
        z = torch.cat([e1, e2], dim=1)
        return self.head(z)

model5 = IntermediateFusionModel(m5_input_dim_fft, num_classes).to(device)
opt5 = torch.optim.Adam(model5.parameters(), lr=0.001)
crit5 = nn.CrossEntropyLoss()

print(model5)


IntermediateFusionModel(
  (imu_enc): IMUTHMEncoder(
    (net): Sequential(
      (0): Linear(in_features=1224, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=64, bias=True)
      (3): ReLU()
    )
  )
  (tof_enc): TOFEncoder(
    (cnn): Sequential(
      (0): Conv2d(5, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Flatten(start_dim=1, end_dim=-1)
      (7): Linear(in_features=128, out_features=64, bias=True)
      (8): ReLU()
    )
    (lstm): LSTM(64, 64, batch_first=True, bidirectional=True)
  )
  (head): Sequential(
    (0): Linear(in_features=192, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_featur

In [18]:
@torch.no_grad()
def compute_norm_stats_m5(loader, max_batches=30, eps=1e-6):
    xs = []
    for i, (x_fft, x_tof, y) in enumerate(loader):
        xs.append(x_fft.float())
        if i + 1 >= max_batches:
            break
    X = torch.cat(xs, dim=0)
    X = torch.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    mu = X.mean(dim=0)
    sigma = X.std(dim=0)
    sigma = torch.clamp(sigma, min=eps)
    return mu, sigma

mu5, sig5 = compute_norm_stats_m5(train_loader_m5, max_batches=30)
mu5, sig5 = mu5.to(device), sig5.to(device)
print("Computed Model 5 FFT normalization stats.")


Computed Model 5 FFT normalization stats.


In [19]:
def train_one_epoch_m5():
    model5.train()
    total_loss, total, correct = 0.0, 0, 0

    for x_fft, x_tof, y in train_loader_m5:
        x_fft = x_fft.to(device).float()
        x_tof = x_tof.to(device).float()
        y = y.to(device)

        # normalize FFT branch only
        x_fft = (x_fft - mu5) / sig5
        x_fft = torch.nan_to_num(x_fft, nan=0.0, posinf=0.0, neginf=0.0)

        opt5.zero_grad()
        logits = model5(x_fft, x_tof)
        loss = crit5(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model5.parameters(), 1.0)
        opt5.step()

        total_loss += loss.item() * y.size(0)
        total += y.size(0)
        correct += (logits.argmax(1) == y).sum().item()

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate_m5():
    model5.eval()
    total, correct = 0, 0
    y_true, y_pred = [], []
    all_logits = []

    for x_fft, x_tof, y in test_loader_m5:
        x_fft = x_fft.to(device).float()
        x_tof = x_tof.to(device).float()
        y = y.to(device)

        x_fft = (x_fft - mu5) / sig5
        x_fft = torch.nan_to_num(x_fft, nan=0.0, posinf=0.0, neginf=0.0)

        logits = model5(x_fft, x_tof)
        pred = logits.argmax(1)

        total += y.size(0)
        correct += (pred == y).sum().item()

        y_true.append(y.cpu().numpy())
        y_pred.append(pred.cpu().numpy())
        all_logits.append(logits.cpu())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    acc = correct / total
    macro = f1_score(y_true, y_pred, average="macro")
    return acc, macro, torch.cat(all_logits, dim=0), torch.tensor(y_true), torch.tensor(y_pred)

EPOCHS_M5 = 20
for ep in range(1, EPOCHS_M5 + 1):
    loss, tr_acc = train_one_epoch_m5()
    te_acc, te_macro, _, _, _ = evaluate_m5()
    print(f"[M5] Epoch {ep:02d} | loss={loss:.4f} | train_acc={tr_acc:.4f} | test_acc={te_acc:.4f} | macroF1={te_macro:.4f}")

# final save
te_acc, te_macro, logits5, y_true5, y_pred5 = evaluate_m5()

torch.save(
    {
        "logits": logits5,
        "y_true": y_true5,
        "y_pred": y_pred5,
        "classes": le.classes_,
        "seq_ids": test_ids,
        "acc": float(te_acc),
        "macro18": float(te_macro)
    },
    DATA_PROCESSED / "model5_intermediate_fusion_outputs.pt"
)
print("Saved model5_intermediate_fusion_outputs.pt")
print("Final Model 5 acc:", te_acc, "| macroF1(18):", te_macro)


[M5] Epoch 01 | loss=2.3701 | train_acc=0.2236 | test_acc=0.2793 | macroF1=0.2221
[M5] Epoch 02 | loss=1.8822 | train_acc=0.3480 | test_acc=0.3395 | macroF1=0.3172
[M5] Epoch 03 | loss=1.6452 | train_acc=0.4121 | test_acc=0.3806 | macroF1=0.3974
[M5] Epoch 04 | loss=1.4994 | train_acc=0.4634 | test_acc=0.3929 | macroF1=0.4194
[M5] Epoch 05 | loss=1.3674 | train_acc=0.5134 | test_acc=0.4058 | macroF1=0.4053
[M5] Epoch 06 | loss=1.2628 | train_acc=0.5429 | test_acc=0.3947 | macroF1=0.4043
[M5] Epoch 07 | loss=1.1518 | train_acc=0.5820 | test_acc=0.3966 | macroF1=0.4146
[M5] Epoch 08 | loss=1.0573 | train_acc=0.6201 | test_acc=0.4033 | macroF1=0.4339
[M5] Epoch 09 | loss=0.9599 | train_acc=0.6566 | test_acc=0.3984 | macroF1=0.4325
[M5] Epoch 10 | loss=0.8528 | train_acc=0.6984 | test_acc=0.4150 | macroF1=0.4463
[M5] Epoch 11 | loss=0.7644 | train_acc=0.7260 | test_acc=0.4064 | macroF1=0.4311
[M5] Epoch 12 | loss=0.7023 | train_acc=0.7489 | test_acc=0.4009 | macroF1=0.4336
[M5] Epoch 13 | 

## Model 6: FFT Random Forest (All Sensors)
We extract FFT vectors (same as Model 1) and train a RandomForestClassifier with 100 trees.


In [20]:
def build_fft_matrix(grouped, seq_ids, feature_cols, max_len):
    X = []
    y = []
    for sid in seq_ids:
        s = grouped[sid]
        X.append(seq_to_fft_vector(s, feature_cols, max_len))
        y.append(le.transform([str(s[LABEL_COL].iloc[0])])[0])
    return np.stack(X), np.array(y)

X_train_rf, y_train_rf = build_fft_matrix(train_groups, train_ids, ALL_SENSORS, MAX_LEN)
X_test_rf,  y_test_rf  = build_fft_matrix(test_groups,  test_ids,  ALL_SENSORS, MAX_LEN)

rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf.fit(X_train_rf, y_train_rf)

rf_pred = rf.predict(X_test_rf)
rf_macro18 = f1_score(y_test_rf, rf_pred, average="macro")
print("RF macro-F1 (18):", rf_macro18)


rf_acc = (rf_pred == y_test_rf).mean()

torch.save(
    {
        "y_true": torch.tensor(y_test_rf),
        "y_pred": torch.tensor(rf_pred),
        "classes": le.classes_,
        "seq_ids": test_ids,
        "acc": rf_acc,
        "macro18": rf_macro18
    },
    DATA_PROCESSED / "model6_fft_random_forest_outputs.pt"
)
print("Saved model6_fft_random_forest_outputs.pt (with metadata)")



RF macro-F1 (18): 0.3812467576590531
Saved model6_fft_random_forest_outputs.pt (with metadata)


## 4.10 Saved Artifacts

The following model outputs are saved for evaluation:

- model1_fft_mlp_all_outputs.pt
- model2_fft_mlp_imu_thm_outputs.pt
- model3_cnn_bilstm_tof_outputs.pt
- model4_late_fusion_outputs.pt
- model5_intermediate_fusion_outputs.pt
- model6_fft_random_forest_outputs.pt

These files are consumed by the evaluation notebook.
